# Manipulasi Data dengan Pandas - Dataset Transaksi POS 2024-2025

Tugas ini mempraktikkan lima materi Pandas pada dataset transaksi POS: **Indexing & Selection**, **Operasi Aritmatika**, **Filtering**, **Sorting**, dan **Grouping & Agregasi**.

In [20]:
import pandas as pd

# Tampilkan angka dengan 2 desimal dan pemisah ribuan
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = None

PATH = "../paket_a_transaksi_pos_2024_2025.csv"
df = pd.read_csv(PATH)

print(df.shape)
df.dtypes

(5030, 9)


,0
id_transaksi,object
tanggal,object
kode_gerai,object
kode_kasir,object
kode_produk,object
jumlah,int64
harga_satuan,object
diskon_persen,int64
status,object


In [21]:
df = pd.read_csv('../paket_a_transaksi_pos_2024_2025.csv')
print('Ukuran data (baris, kolom):', df.shape)
df.head()

Ukuran data (baris, kolom): (5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


Kolom `harga_satuan` terbaca bertipe teks (`object`/`str`), sedangkan kolom lain yang berisi angka sudah numerik. Ini tanda bahwa ada nilai yang formatnya tidak murni angka.

# Task 1 - Indexing & Selection

### 1.1 Pembersihan format harga

Tujuan: mengubah `harga_satuan` menjadi tipe float.

**Mengapa dua langkah?** Di data ini ada dua format yang bermasalah:
- `7500,00` (koma sebagai desimal) -> ganti koma dengan titik, menjadi `7500.00`.
- `13.000` (titik sebagai pemisah **ribuan**) -> jika hanya mengganti koma, nilai ini terbaca `13.0`, padahal maksudnya tiga belas ribu. Karena itu titik ribuan dihapus lebih dulu.

Urutannya penting: titik ribuan dihapus **sebelum** koma diganti titik, supaya `7500,00` tidak ikut terhapus titiknya.

In [22]:
# Contoh nilai bermasalah sebelum dibersihkan
print(df.loc[df["harga_satuan"].astype(str).str.contains(r"[.,]"), "harga_satuan"].head())

df["harga_satuan"] = df["harga_satuan"].astype(str).str.strip()

# 1) Titik pemisah ribuan: "13.000" -> "13000"
df["harga_satuan"] = df["harga_satuan"].str.replace(r"^(\d{1,3})\.(\d{3})$", r"\1\2", regex=True)

# 2) Koma desimal: "7500,00" -> "7500.00", lalu ubah ke float
df["harga_satuan"] = df["harga_satuan"].str.replace(",", ".", regex=False).astype(float)

print("\nTipe data sekarang:", df["harga_satuan"].dtype)
print("Harga terkecil:", df["harga_satuan"].min(), "| terbesar:", df["harga_satuan"].max())

2      7500,00
8     11000,00
19      13.000
51    27000,00
63    13500,00
Name: harga_satuan, dtype: object

Tipe data sekarang: float64
Harga terkecil: 4000.0 | terbesar: 54000.0


### 1.2 Penanganan kuantitas negatif

Nilai `jumlah` yang negatif diubah menjadi positif dengan **Boolean Indexing** + `.loc[]` + `.abs()`.

**Mengapa `.loc[]`?** `.loc[kondisi, kolom]` mengubah langsung data aslinya dan menghindari peringatan `SettingWithCopyWarning` yang muncul jika memakai `df[kondisi]["kolom"] = ...`.

In [23]:
print("Jumlah negatif sebelum:", (df["jumlah"] < 0).sum())

df.loc[df["jumlah"] < 0, "jumlah"] = df.loc[df["jumlah"] < 0, "jumlah"].abs()

print("Jumlah negatif sesudah:", (df["jumlah"] < 0).sum())

Jumlah negatif sebelum: 41
Jumlah negatif sesudah: 0


### 1.3 Imputasi nilai kosong kasir

Nilai kosong (`NaN`) pada `kode_kasir` diisi dengan `K_UNKNOWN` memakai `.loc[]` dan `.isna()`.

**Mengapa diisi, bukan dihapus?** Kolom kasir tidak dipakai menghitung omzet. Jika barisnya dihapus, transaksi yang sah ikut hilang.

In [24]:
print("NaN kode_kasir sebelum:", df["kode_kasir"].isna().sum())

df.loc[df["kode_kasir"].isna(), "kode_kasir"] = "K_UNKNOWN"

print("NaN kode_kasir sesudah:", df["kode_kasir"].isna().sum())

NaN kode_kasir sebelum: 261
NaN kode_kasir sesudah: 0


### 1.4 Seleksi posisi dengan `.iloc[]`

`.iloc[baris, kolom]` memilih berdasarkan **posisi angka** (mulai dari 0). Kolom yang diminta ada di posisi 0 (`id_transaksi`), 1 (`tanggal`), 2 (`kode_gerai`), dan 5 (`jumlah`). Batas akhir irisan `0:5` bersifat eksklusif, jadi yang diambil baris 0 sampai 4 (5 baris).

In [25]:
subset_iloc = df.iloc[0:5, [0, 1, 2, 5]]
subset_iloc

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


# Task 2 - Operasi Aritmatika (Feature Engineering)

Tiga kolom baru dibuat dengan operasi vektor, yaitu perhitungan elemen demi elemen tanpa perulangan.

- `Total_Kotor` = `harga_satuan` x `jumlah`
- `Nominal_Diskon` = `Total_Kotor` x `diskon_persen / 100` (dibagi 100 karena skalanya 0-100)
- `Total_Bersih` = (`Total_Kotor` - `Nominal_Diskon`) x 1,11 (PPN 11% dikenakan pada nilai **setelah** diskon)

In [26]:
df["Total_Kotor"] = df["harga_satuan"] * df["jumlah"]
df["Nominal_Diskon"] = df["Total_Kotor"] * (df["diskon_persen"] / 100)
df["Total_Bersih"] = (df["Total_Kotor"] - df["Nominal_Diskon"]) * 1.11

df[["harga_satuan", "jumlah", "diskon_persen", "Total_Kotor", "Nominal_Diskon", "Total_Bersih"]].head(10)

,harga_satuan,jumlah,diskon_persen,Total_Kotor,Nominal_Diskon,Total_Bersih
0,"13,000.00",4,0,"52,000.00",0.00,"57,720.00"
1,"21,000.00",1,0,"21,000.00",0.00,"23,310.00"
2,"7,500.00",3,0,"22,500.00",0.00,"24,975.00"
3,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"
4,"6,000.00",1,0,"6,000.00",0.00,"6,660.00"
5,"45,500.00",12,10,"546,000.00","54,600.00","545,454.00"
6,"18,000.00",1,0,"18,000.00",0.00,"19,980.00"
7,"47,500.00",4,10,"190,000.00","19,000.00","189,810.00"
8,"11,000.00",3,0,"33,000.00",0.00,"36,630.00"
9,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"


# Task 3 - Filtering (Penyaringan Data)

### 3.1 Filter multi-kondisi

Tiga syarat digabung dengan operator `&`: status `SELESAI`, gerai G05 **atau** G09, dan `Total_Bersih` di atas Rp 100.000.

- Setiap kondisi **wajib dikurung** `( )` karena `&` punya prioritas lebih tinggi daripada operator perbandingan.
- `.isin(["G05", "G09"])` dipakai sebagai pengganti `OR` agar lebih ringkas.

In [27]:
filter_1 = df[
    (df["status"] == "SELESAI") &
    (df["kode_gerai"].isin(["G05", "G09"])) &
    (df["Total_Bersih"] > 100000)
]

print("Jumlah baris yang memenuhi kriteria:", len(filter_1))
filter_1.head(10)

Jumlah baris yang memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"
213,TRX0244651,2025-04-15,G09,K010,PRW1303,2,"46,000.00",0,SELESAI,"92,000.00",0.00,"102,120.00"
216,TRX0241198,2024-06-16,G09,K020,BYI1502,6,"28,000.00",0,SELESAI,"168,000.00",0.00,"186,480.00"
220,TRX0241165,2025-09-24,G09,K023,SGR1208,6,"27,500.00",0,SELESAI,"165,000.00",0.00,"183,150.00"
373,TRX0240601,2025-04-28,G05,K040,SGR1204,12,"50,000.00",0,SELESAI,"600,000.00",0.00,"666,000.00"
410,TRX0241526,2024-12-31,G05,K026,BYI1501,6,"18,500.00",0,SELESAI,"111,000.00",0.00,"123,210.00"


### 3.2 Filter dengan `.query()`

`.query()` menulis kondisi sebagai teks bergaya SQL, sehingga tidak perlu tanda kurung berlebih.

In [28]:
filter_g10 = df.query("kode_gerai == 'G10' and jumlah >= 5")

print("Jumlah baris:", len(filter_g10))
filter_g10.head(10)

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,"52,000.00",10,SELESAI,"312,000.00","31,200.00","311,688.00"
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,"18,000.00",0,SELESAI,"108,000.00",0.00,"119,880.00"
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,"6,000.00",0,SELESAI,"36,000.00",0.00,"39,960.00"
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,"54,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,"29,000.00",0,SELESAI,"174,000.00",0.00,"193,140.00"


# Task 4 - Sorting (Pengurutan Data)

### 4.1 Pengurutan multi-kolom

`sort_values()` menerima daftar kolom dan daftar arah: `kode_gerai` naik (A-Z, `True`), lalu `Total_Bersih` turun (`False`). Urutan kolom di `by` menentukan prioritas.

In [29]:
df_urut = df.sort_values(by=["kode_gerai", "Total_Bersih"], ascending=[True, False])
df_urut[["id_transaksi", "kode_gerai", "Total_Bersih"]].head(10)

,id_transaksi,kode_gerai,Total_Bersih
244,TRX0243257,G01,"42,020,160.00"
2956,TRX0241367,G01,"672,660.00"
3588,TRX0240608,G01,"626,040.00"
2937,TRX0240631,G01,"559,440.00"
5,TRX0240658,G01,"545,454.00"
3889,TRX0241537,G01,"359,640.00"
2179,TRX0244635,G01,"349,650.00"
4270,TRX0240428,G01,"323,010.00"
695,TRX0244706,G01,"313,020.00"
4404,TRX0242179,G01,"313,020.00"


### 4.2 Top-N dengan `.nlargest()`

`.nlargest(5, "Total_Bersih")` mengambil 5 baris dengan nilai terbesar tanpa harus mengurutkan seluruh tabel.

**Catatan:** 5 transaksi teratas semuanya memiliki `jumlah` ribuan unit (misalnya 2.093 dan 1.301), padahal jumlah wajar per transaksi hanya 1-12. Ini kemungkinan **kesalahan input** yang terbaca sebagai "transaksi terbesar".

In [30]:
top_5 = df.nlargest(5, "Total_Bersih")
top_5[["id_transaksi", "kode_gerai", "jumlah", "harga_satuan", "status", "Total_Bersih"]]

,id_transaksi,kode_gerai,jumlah,harga_satuan,status,Total_Bersih
994,TRX0241002,G11,2093,"48,000.00",SELESAI,"100,363,536.00"
4985,TRX0244221,G09,1301,"51,500.00",SELESAI,"66,934,498.50"
467,TRX0243696,G06,892,"54,000.00",SELESAI,"53,466,480.00"
3393,TRX0241029,G07,1983,"23,000.00",SELESAI,"45,563,391.00"
244,TRX0243257,G01,1456,"32,500.00",SELESAI,"42,020,160.00"


# Task 5 - Grouping & Agregasi

### 5.1 Laporan agregasi per gerai

Data disaring dulu hanya untuk status `SELESAI`, lalu dikelompokkan per `kode_gerai` (paradigma *split-apply-combine*). `.agg()` dengan **named aggregation** membuat beberapa metrik sekaligus dengan nama kolom sendiri. Terakhir diurutkan dari omzet tertinggi dan dibulatkan 2 desimal.

In [31]:
selesai = df[df["status"] == "SELESAI"]

laporan_gerai = (
    selesai.groupby("kode_gerai", as_index=False)
    .agg(
        Jumlah_Transaksi=("id_transaksi", "count"),
        Total_Unit_Terjual=("jumlah", "sum"),
        Total_Omzet_Bersih=("Total_Bersih", "sum"),
        Rata_Rata_Transaksi=("Total_Bersih", "mean"),
    )
    .sort_values(by="Total_Omzet_Bersih", ascending=False)
    .round(2)
)
laporan_gerai

,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
10,G11,398,3170,"125,638,152.75","315,673.75"
8,G09,379,3027,"101,157,630.00","266,906.68"
5,G06,383,3103,"91,228,624.50","238,194.84"
6,G07,384,4284,"80,559,748.50","209,791.01"
0,G01,450,2635,"70,967,378.25","157,705.28"
2,G03,372,3274,"59,991,226.50","161,266.74"
9,G10,401,1742,"58,964,892.75","147,044.62"
7,G08,408,1700,"35,630,084.25","87,328.64"
11,G12,406,1010,"24,699,220.50","60,835.52"
3,G04,399,1042,"24,666,503.25","61,820.81"


### 5.2 Transformasi kelompok (`.transform()`)

Berbeda dengan `.agg()` yang meringkas tabel, `.transform("sum")` menghitung total per gerai lalu **mengembalikannya ke setiap baris**, sehingga panjang hasilnya sama dengan tabel utama. Dengan begitu total gerai bisa langsung dipakai sebagai pembagi: `(Total_Bersih / total gerai) x 100`.

Perhitungan ini memakai seluruh transaksi di `df`. Cek: jumlah kontribusi tiap gerai harus 100%.

In [32]:
total_per_gerai = df.groupby("kode_gerai")["Total_Bersih"].transform("sum")
df["Kontribusi_Gerai_Persen"] = df["Total_Bersih"] / total_per_gerai * 100

# Validasi: tiap gerai harus berjumlah 100%
print(df.groupby("kode_gerai")["Kontribusi_Gerai_Persen"].sum().round(6).unique())

df[["id_transaksi", "kode_gerai", "Total_Bersih", "Kontribusi_Gerai_Persen"]].head(20)

[100.]


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,"57,720.00",0.06
1,TRX0240510,G09,"23,310.00",0.02
2,TRX0244483,G11,"24,975.00",0.02
3,TRX0242532,G05,"14,430.00",0.06
4,TRX0243509,G05,"6,660.00",0.03
5,TRX0240658,G01,"545,454.00",0.75
6,TRX0244581,G10,"19,980.00",0.03
7,TRX0241251,G09,"189,810.00",0.19
8,TRX0244274,G07,"36,630.00",0.04
9,TRX0243163,G08,"14,430.00",0.04


## Catatan

- Pembersihan format harga, jumlah negatif, dan kasir kosong. Baris duplikat (30 baris) dan `jumlah` ekstrem (12 baris di atas 12 unit) **tidak dihapus** karena tidak diminta.
- Titik ribuan pada `harga_satuan` (misalnya `13.000`) ditangani terpisah. Jika hanya mengganti koma, 45 harga akan terbaca keliru (misalnya 13 rupiah).
- Kontribusi gerai (5.2) dihitung dari seluruh transaksi, sedangkan laporan gerai (5.1) hanya dari status `SELESAI`.